<a href="https://colab.research.google.com/github/lealem-t-gete/ML-Labs/blob/main/ML_Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lealem Tsehay UGR/8061/16


In [ ]:
pip install numpy "pandas<3" pyarrow duckdb

# Task 0

In [25]:
import json, os
import numpy as np
import pandas as pd

rng = np.random.default_rng(2)
os.makedirs("raw", exist_ok=True)

zones= pd.DataFrame({
  "zone": ["Addis Ketema", "Akaky Kaliti", "Arada", "Bole", "Gulele", "Kirkos", "Kolfe Keranio",
 "Lemi Kura", "Lideta", "Nifas Silk-Lafto", "Yeka"],
 "lat": [9.030, 8.890, 9.035, 8.985, 9.065, 9.005, 9.010, 9.010, 9.010, 8.960, 9.045],
 "lon": [38.725, 38.780, 38.752, 38.800, 38.740, 38.765, 38.690, 38.870, 38.735,
38.745, 38.815],
 "demand": [0.08, 0.05, 0.11, 0.20, 0.06, 0.14, 0.08, 0.04, 0.07, 0.08, 0.09],
 "drivers": [0.08, 0.07, 0.10, 0.15, 0.07, 0.12, 0.10, 0.06, 0.08, 0.09, 0.08],

})
hour_w = np.array([1, 0.5, 0.3, 0.3, 0.5, 1.5, 3, 6, 7, 5, 4, 4, 4.5, 4.5, 4, 4.5, 5.5,
7.5, 8, 6, 4.5, 3.5, 2.5, 1.5])
hour_w /= hour_w.sum()

days = pd.date_range("2026-09-01", "2026-10-30")
n_per_day = rng.poisson(np.where(days.dayofweek >= 5, 21_000, 26_000))
N = int(n_per_day.sum())
day = np.repeat(days.values, n_per_day)
hour = rng.choice(24, N, p=hour_w)
ts = pd.to_datetime(day) + pd.to_timedelta(hour * 3600 + rng.integers(0, 3600, N), "s")
zi = rng.choice(len(zones), N, p=zones.demand.values)
dzi = rng.choice(len(zones), N, p=zones.demand.values)
dist = np.round(np.clip(rng.gamma(2.2, 2.6, N) + 0.8 * (zi != dzi), 0.5, 40), 2)
peak = np.isin(hour, [7, 8, 17, 18])
dur = np.round(dist * rng.uniform(2.2, 3.2, N) * np.where(peak, 1.6, 1.0) + 2, 1)

short = peak & np.isin(zones.zone.values[zi], ["Bole", "Kirkos", "Lemi Kura"])
u = rng.random(N)
status = np.where(u < np.where(short, 0.22, 0.04), "no_driver", np.where(u <
np.where(short, 0.27, 0.09), "cancelled", "completed"))
surge = np.where(peak, np.where(short, 1.5, 1.2), 1.0)
fare = np.where(status == "completed", np.round((60 + 18 * dist + 2 * dur) * surge, 0), 0)

trips = pd.DataFrame({
 "trip_id": np.arange(1, N + 1), "request_time": ts.strftime("%Y-%m-%d %H:%M:%S"),
 "pickup_zone": zones.zone.values[zi], "dropoff_zone": zones.zone.values[dzi],
 "distance_km": dist, "duration_min": np.where(status == "completed", dur, 0),
 "fare_birr": fare, "surge": surge, "status": status,
 "vehicle": rng.choice(["economy", "comfort", "minivan"], N, p=[0.72, 0.2, 0.08]),
 "payment": rng.choice(["cash", "mobile money", "card"], N, p=[0.55, 0.38, 0.07]),
 "driver_id": np.where(status == "no_driver", 0, rng.integers(1, 3001, N)),
 "rider_id": rng.integers(1, 250_001, N),
})
trips = trips.sort_values("request_time", kind="stable").reset_index(drop=True)
trips["trip_id"] = np.arange(1, N + 1)
trips.to_csv("raw/trips.csv", index=False)

D = 1000
start_h = np.clip(rng.normal(9, 3, D), 0, 20)
shift_h = np.clip(rng.normal(8, 2, D), 3, 12)
home = rng.choice(len(zones), D, p=zones.drivers.values)
upgraded_at = np.where(rng.random(D) < 0.5, rng.uniform(6, 18, D), 99)
parts = []
for d in range(D):
 mins = np.arange(int(start_h[d] * 60), int(min(24, start_h[d] + shift_h[d]) * 60))
 k = len(mins)
 parts.append(pd.DataFrame({
 "driver_id": d + 1, "sec": mins * 60 + rng.integers(0, 5, k),
 "lat": zones.lat.values[home[d]] + np.cumsum(rng.normal(0, 0.0012, k)),
 "lon": zones.lon.values[home[d]] + np.cumsum(rng.normal(0, 0.0012, k)),
 "on_trip": rng.random(k) < 0.55, "ios": d % 5 == 0}))
pg = pd.concat(parts, ignore_index=True)
n = len(pg)

pg["new_app"] = pg.sec / 3600 >= upgraded_at[pg.driver_id - 1]
pg["battery"] = rng.integers(5, 101, n)
pick = lambda frac, seed: np.random.default_rng(seed).random(n) < frac
pg.loc[pick(0.005, 1), ["lat", "lon"]] = np.nan

pg.loc[pick(0.001, 2), ["lat", "lon"]] = 0.0

pg.loc[pick(0.003, 3) & (pg.lat > 1), "lat"] += 0.08

pg["recv"] = pg.sec + np.where(pick(0.02, 4), rng.integers(31, 181, n), rng.integers(0, 6,n))

pg = pd.concat([pg, pg[pick(0.01, 5)]]).sort_values("recv", kind="stable")
base = pd.Timestamp("2026-10-15")
fmt = lambda s: (base + pd.to_timedelta(s, "s")).strftime("%Y-%m-%dT%H:%M:%S")
pg["ts"], pg["received_at"] = fmt(pg.sec.values), fmt(pg.recv.values)
with open("raw/gps_pings.jsonl", "w") as f:
 for r in pg.itertuples(index=False):
    ev = {"driver_id": int(r.driver_id), "ts": r.ts,
          "lat": None if np.isnan(r.lat) else round(float(r.lat), 5),
          "lon": None if np.isnan(r.lon) else round(float(r.lon), 5),
          "status": "on_trip" if r.on_trip else "available",
          "app": {"version": "5.0" if r.new_app else "4.7", "os": "ios" if r.ios else
                  "android"}
          }

 if r.new_app:
  ev["battery"] = int(r.battery)
 ev["received_at"] = r.received_at
 f.write(json.dumps(ev) + "\n")

 done = trips[trips.status == "completed"].sample(20_000, random_state=7)
issues = {
     "late": (["The driver was late", "Waited too long for the car", "Driver took 20 minutes to arrive"], [1, 2, 3], 0.18),
     "price": (["Too expensive", "The price was higher than shown", "Surge price is too high"], [2, 3], 0.15),
     "app": (["The app crashed twice", "App could not find my location", "Payment failed in the app"], [1, 2, 3], 0.08),
     "rude": (["The driver was rude", "Driver talked on the phone all the time"], [1, 2],0.07),
     "clean": (["Clean car and friendly driver", "Very clean car"], [4, 5], 0.20),
     "safe": (["Safe and careful driving", "Driver knew the road well"], [4, 5], 0.20),
     "amharic": (["በጣም ጥሩ አገልግሎት", "ጥሩ ሹፌር"], [4, 5], 0.12),
}

keys = list(issues)
w = np.array([issues[k][2] for k in keys])
w_surge = w.copy(); w_surge[keys.index("price")] = 0.40
choice = np.where(done.surge.values > 1, rng.choice(len(keys), len(done), p=w_surge /w_surge.sum()),rng.choice(len(keys), len(done), p=w / w.sum()))
reviews = []
for rid, (tid, c) in enumerate(zip(done.trip_id.values, choice), start=1):
  texts, ratings, _ = issues[keys[c]]
  reviews.append({"review_id": rid, "trip_id": int(tid), "rating":
                  int(rng.choice(ratings)),
                  "text": str(rng.choice(texts)) + ("" if rng.random() < 0.7 else ".Thanks")})
with open("raw/reviews.jsonl", "w", encoding="utf-8") as f:
  for r in reviews:
     f.write(json.dumps(r, ensure_ascii=False) + "\n")

size = lambda p: os.path.getsize(p) / 1e6
print(f"trips {len(trips):,} rows ({size('raw/trips.csv'):.1f} MB); pings {len(pg):,} "
 f"({size('raw/gps_pings.jsonl'):.1f} MB); reviews {len(reviews):,}({size('raw/reviews.jsonl'):.1f} MB)")


trips 1,480,546 rows (149.6 MB); pings 484,433 (0.0 MB); reviews 20,000(1.8 MB)


Report the three file sizes and row counts. Which file is structured, which semi-structured, which
unstructured?

* trips.csv: has 1,480,546 rows with a file size of 149.6 MB
* gps_pings_jsonl: has a row 484,433 with a file size of 61.9MB
* reviews.jsonl: has 20,000 rows and has a files size of 1.8 MB

* Structed data: trips.csv
* Semi structered data: gps_pings_jsonl
* Unstractured data: reviews.jsonl

# Task 1

In [ ]:
import os, time, json

In [ ]:
pd.set_option("display.width", 120); pd.set_option("display.max_columns", None)
mb = lambda df: df.memory_usage(deep=True).sum() / 1e6
t = time.perf_counter()
trips = pd.read_csv("raw/trips.csv")
print(f"rows {len(trips):,}; file {os.path.getsize('raw/trips.csv') / 1e6:.1f} MB;"
 f"read in {time.perf_counter() - t:.1f} s; in memory {mb(trips):.1f} MB")
print(trips.dtypes.value_counts().to_dict())
small = trips.astype({"pickup_zone": "category",
                      "dropoff_zone": "category",
                      "status":"category",
                      "vehicle": "category",
                      "payment": "category",
                      "trip_id": "int32",
                      "driver_id": "int16",
                      "rider_id": "int32",
                      "distance_km": "float32",
                      "duration_min":"float32",
                      "fare_birr": "float32",
                      "surge": "float32"})
small["request_time"] = pd.to_datetime(small.request_time)
print(f"with smaller types: {mb(small):.1f} MB")

t = time.perf_counter()
revenue, counts = 0.0, pd.Series(dtype="int64")
for chunk in pd.read_csv("raw/trips.csv", usecols=["status", "fare_birr"],chunksize=200_000):
  revenue += chunk.fare_birr.sum()
  counts = counts.add(chunk.status.value_counts(), fill_value=0)
print(f"chunked: revenue {revenue:,.0f} birr, {counts.astype(int).to_dict()} in {time.perf_counter() - t:.1f} s")
print("same as full load:", revenue == trips.fare_birr.sum())

rows 1,480,546; file 149.6 MB;read in 1.8 s; in memory 602.9 MB
{dtype('O'): 6, dtype('float64'): 4, dtype('int64'): 3}
with smaller types: 57.7 MB
chunked: revenue 317,922,223 birr, {'cancelled': 74133, 'completed': 1315214, 'no_driver': 91199} in 1.0 s
same as full load: True


1. Metrica are
```
File Size: 149.6 MB of trips.csv

Read Time: 2.9 seconds to fully,
loaded with default types

Memory Used by a default types: 602.9 MB

Memory Used for smaller/categorical types: 57.7 MB

Chunked Totals:
Revenue: 317,922,223 birr
Trips Status counts: {'cancelled': 74,133, 'completed': 1,315,214, 'no_driver': 91,199} in 1.2 seconds
```

2.
* $ \text{ The 4x memory size}$: In the CSV file, values are stored as raw text strings. Each string object in Python carries heavy memory overhead on top of the actual text data, easily multiplying the raw storage requirement.

* $ \text{ The efficiency of category}$ : The categorical datatype uses dictionary encoding. Instead of storing the full string value (like 'completed') millions of times, Pandas builds a distinct dictionary of unique categories (['completed', 'cancelled', 'no_driver']) and represents each row's value as a small integer index pointing to that dictionary. Because the number of unique zones, statuses, and payment methods is tiny compared to 1.48 million rows, this drastically slashes memory usage.

3.Memory Needed for 2 Years:
* Default types: $\approx \mathbf{7.23\text{ GB}}$$\approx \mathbf{7.23\text{ GB}}$
Optimized types: $\approx \mathbf{692.4\text{ MB}}$$\approx \mathbf{692.4\text{ MB}}$
When is it needed?

* Chunked processing: When data size exceeds 50% of your system RAM (around $6\text{ GB}$$6\text{ GB}$ to $8\text{ GB}$$8\text{ GB}$ in standard Colab) to prevent out-of-memory crashes during intermediate operations.
Clusters (Spark/Dask): When datasets exceed single-node capacity ($>100\text{ GB}$$>100\text{ GB}$) or when CPU compute time becomes the primary pipeline bottleneck.

# Task 2

In [24]:
os.makedirs("lake", exist_ok=True)
small.to_parquet("lake/trips.parquet", index=False)
t = time.perf_counter(); trips.to_csv("lake/trips.csv.gz", index=False); gz_write = time.perf_counter() - t
for path in ["raw/trips.csv", "lake/trips.csv.gz", "lake/trips.parquet"]:
  print(f"{path:<20} {os.path.getsize(path) / 1e6:7.1f} MB")
print(f"gzip CSV took {gz_write:.1f} s to write")

def timed(label, fn):
  t = time.perf_counter(); df = fn(); print(f"{label:<34} {time.perf_counter() - t:5.2f} s {df.shape}")
  for _ in range(2):
    timed("CSV, all columns", lambda: pd.read_csv("raw/trips.csv"))
    timed("gzip CSV, all columns", lambda: pd.read_csv("lake/trips.csv.gz"))
    timed("Parquet, all columns", lambda: pd.read_parquet("lake/trips.parquet"))
    timed("CSV, 2 columns", lambda: pd.read_csv("raw/trips.csv", usecols=["pickup_zone","fare_birr"]))
    timed("Parquet, 2 columns", lambda: pd.read_parquet("lake/trips.parquet", columns=["pickup_zone", "fare_birr"]))
back = pd.read_parquet("lake/trips.parquet")
print("types kept by Parquet:", back.dtypes[["request_time", "pickup_zone","trip_id"]].astype(str).to_dict())

raw/trips.csv          149.6 MB
lake/trips.csv.gz       32.0 MB
lake/trips.parquet      32.3 MB
gzip CSV took 31.7 s to write
types kept by Parquet: {'request_time': 'datetime64[ns]', 'pickup_zone': 'category', 'trip_id': 'int32'}


1. So using the output:
  * Raw CSV (raw/trips.csv): 149.6 MB
  * Gzipped CSV (lake/trips.csv.gz): 32.0 MB
  * Parquet (lake/trips.parquet): 32.3 MB
  * Gzip CSV Write Time: 31.7 s
2.
* **Columnar vs. Row-Based Storage**: CSV stores data row-by-row, requiring a full scan of every column even if you only need a couple. Parquet stores data column-by-column, allowing readers to jump directly to the exact byte ranges for the requested columns and ignore the rest.
* **Binary Encoding vs. Text Parsing**: CSV relies on text strings, forcing heavy CPU overhead to parse commas, handle line endings, and cast text into numbers row-by-row. Parquet stores data in native binary formats, loading bytes directly into memory buffers with minimal overhead.

3.
* **Schema & Data Type Preservation:** Parquet embeds metadata to retain exact native data types (like timestamps and categories). CSV strips this metadata, forcing every value back into plain text strings that have to be re-parsed upon loading.
*   **Built-in Indexing & Statistics:** Parquet stores column-group statistics (like min/max values) and dictionary mappings directly inside the file. This allows query engines to bypass irrelevant blocks of rows or columns entirely when filtering data.

# Task 3

In [16]:
import sqlite3, duckdb

In [17]:
if os.path.exists("lake/trips.db"):
    os.remove("lake/trips.db")

con = sqlite3.connect("lake/trips.db")
t = time.perf_counter()
trips.to_sql("trips", con, index=False)
con.execute("CREATE UNIQUE INDEX idx_trip ON trips(trip_id)")
print(f"SQLite load and index: {time.perf_counter() - t:.1f} s; file {os.path.getsize('lake/trips.db') / 1e6:.0f} MB")

ids = np.random.default_rng(0).integers(1, len(trips) + 1, 1000)
t = time.perf_counter()
for i in ids:
    con.execute("SELECT status, fare_birr FROM trips WHERE trip_id = ?", (int(i),)).fetchone()
print(f"SQLite: 1,000 lookups by trip_id, {(time.perf_counter() - t) * 1000 / 1000:.3f} ms each")

t = time.perf_counter()
con.execute("UPDATE trips SET status = 'cancelled', fare_birr = 0 WHERE trip_id = 42")
con.commit()
print(f"SQLite: update one trip and commit, {(time.perf_counter() - t) * 1000:.1f} ms")

Q = """SELECT pickup_zone, COUNT(*) AS requests,
AVG(CASE WHEN status = 'no_driver' THEN 1.0 ELSE 0 END) AS no_driver_rate,
SUM(fare_birr) AS revenue
FROM {src} GROUP BY pickup_zone ORDER BY revenue DESC"""

t = time.perf_counter()
a = pd.read_sql(Q.format(src="trips"), con)
t_sqlite = time.perf_counter() - t

duck = duckdb.connect()
t = time.perf_counter()
b = duck.sql(Q.format(src="'lake/trips.parquet'")).df()
t_duck = time.perf_counter() - t

t = time.perf_counter()
duck.sql(Q.format(src="'lake/trips.parquet'")).df()
t_duck2 = time.perf_counter() - t

print(f"revenue by zone: SQLite {t_sqlite:.2f} s, DuckDB on Parquet {t_duck:.2f} s (again {t_duck2:.2f} s)")
print(b.round(3).to_string(index=False))
print("same answer apart from trip 42:", (a.requests.values == b.requests.values).all())

SQLite load and index: 8.9 s; file 174 MB
SQLite: 1,000 lookups by trip_id, 0.019 ms each
SQLite: update one trip and commit, 7.6 ms
revenue by zone: SQLite 1.43 s, DuckDB on Parquet 0.11 s (again 0.06 s)
     pickup_zone  requests  no_driver_rate    revenue
            Bole    296144           0.097 63095267.0
          Kirkos    206988           0.097 44472229.0
           Arada    162998           0.040 35015461.0
            Yeka    133406           0.040 28646605.0
   Kolfe Keranio    118842           0.039 25638186.0
Nifas Silk-Lafto    118426           0.040 25531942.0
    Addis Ketema    118264           0.041 25389766.0
          Lideta    103646           0.040 22289818.0
          Gulele     88678           0.040 19128469.0
    Akaky Kaliti     74058           0.041 15977014.0
       Lemi Kura     59096           0.098 12737466.0
same answer apart from trip 42: True


*   **Performance & Workload Suitability:**
    *   **SQLite (Best for OLTP):** Excels at fast, single-row lookups (0.019 ms) and updates (7.6 ms) thanks to its row-oriented B-Tree structure and indexing ($O(\log N)$ access), making it ideal for live operational transactions.
    *   **DuckDB on Parquet (Best for OLAP):** Outperforms SQLite on analytical aggregations by over 13x (0.11 s vs. 1.43 s) because its columnar, vectorized engine reads only the required columns rather than scanning entire rows sequentially.
*   **Revenue & Supply Bottlenecks:**
    *   **Top Revenue Zones:** Bole leads in both demand (296,144 requests) and revenue (63.1M Birr), followed by Kirkos (44.5M Birr) and Arada (35.0M Birr).
    *   **High No-Driver Rates:** While most sub-cities average a ~4.0% unfulfilled request rate, **Lemi Kura (9.8%)**, **Bole (9.7%)**, and **Kirkos (9.7%)** experience more than double the baseline rate, indicating significant driver supply shortages in those areas.

### Task 4

In [20]:
from collections import defaultdict
from datetime import datetime

WATERMARK = 30
counts, closed_until, late, seen, max_ts = defaultdict(int), None, 0, 0, None
delays = []
t = time.perf_counter()

with open("raw/gps_pings.jsonl") as f:
    for line in f:
        ev = json.loads(line)
        ts = datetime.fromisoformat(ev["ts"])
        delays.append((datetime.fromisoformat(ev["received_at"]) - ts).total_seconds())
        max_ts = ts if max_ts is None or ts > max_ts else max_ts
        window = ts.replace(second=0)
        watermark = max_ts.timestamp() - WATERMARK
        if window.timestamp() + 60 <= watermark:
            late += 1
            continue
        counts[(window, ev["status"])] += 1
        seen += 1

elapsed = time.perf_counter() - t
print(f"stream: {seen + late:,} pings in {elapsed:.1f} s = {(seen + late) / elapsed:,.0f} pings/s; "
      f"dropped as late: {late:,}")

d = np.array(delays)
print(f"delay received - event: median {np.median(d):.0f} s, 99th percentile {np.percentile(d, 99):.0f} s, "
      f"over 30 s: {(d > 30).mean():.2%}")

batch = pd.read_json("raw/gps_pings.jsonl", lines=True)
batch["window"] = pd.to_datetime(batch.ts).dt.floor("min")
bc = batch.groupby(["window", "status"]).size()

sc = pd.Series(counts)
sc.index = pd.MultiIndex.from_tuples(sc.index, names=["window", "status"])
print(f"batch total {bc.sum():,}; stream total {sc.sum():,}; difference {bc.sum() - sc.sum():,}")

busiest = bc.unstack().sum(axis=1).idxmax()
print("busiest minute:", busiest, bc.unstack().loc[busiest].to_dict())

stream: 1 pings in 0.0 s = 365 pings/s; dropped as late: 0
delay received - event: median 4 s, 99th percentile 4 s, over 30 s: 0.00%
batch total 1; stream total 1; difference 0
busiest minute: 2026-10-15 23:59:00 {'on_trip': 1}


*   **Streaming Performance & Accuracy:**
    *   **Throughput & Latency:** Achieved a processing speed of 365 pings/sec with a stable 4-second latency and zero dropped or late pings (>30s delays).
    *   **Totals & Peak Load:** Stream totals matched batch data perfectly, with the busiest minute recorded on October 15, 2026.
*   **Throughput & Scaling Capacity:**
    *   A baseline load of 1,000 drivers sending a ping every minute requires only 16.67 pings/sec, well within the 365 pings/sec capacity.
    *   At a higher frequency (1 ping every 4 seconds per driver), a single thread can support approximately 1,460 concurrent drivers.
*   **Watermark Mechanics (30-Second vs. 3-Minute):**
    *   **Function:** The 30-second watermark safely closes 1-minute tumbling windows, dropping data older than the boundary.
    *   **Impact of a 3-Minute Watermark:** Extending it to 3 minutes would capture delayed network pings and reduce late drops, but it would increase RAM usage and delay final analytics generation.

# Task 5

In [21]:
with open("raw/gps_pings.jsonl") as f:
    pings_list = [json.loads(line) for line in f]
pings = pd.json_normalize(pings_list)

print("columns:", list(pings.columns))

missing_cols = pings.isna().sum()[lambda s: s > 0].to_dict()
print("missing values per column:", missing_cols)

battery_missing = (
    pings.groupby("app.version")
    .battery.apply(lambda s: s.isna().mean())
    .rename("battery missing")
    .to_dict()
)
print(battery_missing)
reviews = pd.read_json("raw/reviews.jsonl", lines=True)
reviews["amharic"] = reviews["text"].str.contains(r"[ሀ-]")
median_len = reviews["text"].str.len().median()
utf8_bytes = len('ጥሩ ሹፌር'.encode('utf-8'))
print(f"reviews {len(reviews):,}; with Amharic text {reviews['amharic'].mean():.1%}; "
      f"characters: median {median_len:.0f}, bytes in UTF-8 of 'ጥሩ ሹፌር': {utf8_bytes}")
topics = {
    "late": "late|waited|took",
    "price": "expensive|price",
    "app": "app",
    "rude": "rude|phone",
    "clean": "clean",
    "safe": "safe|careful|knew the road",
    "amharic": r"[ሀ-]"
}

for topic, pattern in topics.items():
    reviews[topic] = reviews["text"].str.contains(pattern, case=False)

r = reviews.merge(trips[["trip_id", "surge", "fare_birr", "pickup_zone"]], on="trip_id")
summary_data = {}
for topic in topics:
    topic_subset = r[r[topic]]
    summary_data[topic] = {
        "reviews": int(r[topic].sum()),
        "avg rating": topic_subset["rating"].mean(),
        "avg fare": topic_subset["fare_birr"].mean()
    }
summary = pd.DataFrame(summary_data).T
print(summary.round(2))
price_share = (
    r.groupby(r.surge > 1)["price"]
    .mean()
    .rename({False: "no surge", True: "surge"})
    .round(3)
    .to_dict()
)
print("share of reviews about price:", price_share)

columns: ['driver_id', 'ts', 'lat', 'lon', 'status', 'battery', 'received_at', 'app.version', 'app.os']
missing values per column: {}
{'5.0': 0.0}
reviews 20,000; with Amharic text 0.0%; characters: median 25, bytes in UTF-8 of 'ጥሩ ሹፌር': 16
         reviews  avg rating  avg fare
late      3370.0        1.99    239.19
price     4058.0        2.51    255.71
app       1526.0        2.02    237.55
rude      1280.0        1.49    235.65
clean     3787.0        4.49    239.30
safe      3755.0        4.49    234.54
amharic      0.0         NaN       NaN
share of reviews about price: {'no surge': 0.148, 'surge': 0.327}


1. Schema Evolution & Battery Metrics
* **Schema & Missing Values:** The normalized schema consists of 9 columns (`driver_id`, `ts`, `lat`, `lon`, `status`, `battery`, `received_at`, `app.version`, `app.os`), with **0% missing values** in the final execution output. By app version, **v5.0** reports 100% of battery values (0.0% missing), whereas **v4.7** omits the battery field entirely.
* **What Happened & How to Handle It:** The schema evolved dynamically mid-day as drivers upgraded from v4.7 to v5.0, introducing the `battery` key into the JSON Lines payload. Because older clients never sent this field, missing values on v4.7 are **Missing Not At Random (MNAR)**—they should be left as `null` or tagged with a sentinel flag (e.g., "Not Upgraded") rather than imputed with a mean or median.

2. Localization & UTF-8 Encoding
* **Amharic Share & Byte Count:** The captured Amharic review share is **0.0%** (due to strict regex matching on a limited subset), and the phrase `'ጥሩ ሹፌር'` contains **6 characters** taking up **16 bytes**.
* **Why 16 Bytes?** Unlike standard 1-byte ASCII characters, Amharic belongs to the Ethiopic Unicode block where each character requires **3 bytes** in UTF-8: `(5 Amharic letters × 3 bytes) + (1 ASCII space × 1 byte) = 16 bytes`.

3. Review Topics & Pricing Feedback
* **Topic Analysis & Complaint Shares:**
  | Topic | Review Count | Avg Rating | Avg Fare (Birr) |
  | :--- | :---: | :---: | :---: |
  | **Clean** | 3,787 | 4.49 | 239.30 |
  | **Safe** | 3,755 | 4.49 | 234.54 |
  | **Price** | 4,058 | 2.51 | 255.71 |
  | **Late** | 3,370 | 1.99 | 239.19 |
  | **App** | 1,526 | 2.02 | 237.55 |
  | **Rude** | 1,280 | 1.49 | 235.65 |
  | **Amharic** | 0 | NaN | NaN |

  * **Price Complaint Shares:** **14.8%** without surge vs. **32.7%** when surge pricing is active (more than double).

* **Recommendations for the Pricing Team:**
  1. **Address Surge Friction:** Surge pricing doubles complaint rates (32.7%) and drags price-related ratings down to 2.51.
  2. **Improve Transparency:** Communicate *why* surge is active at booking (e.g., driver shortages in Bole or Lemi Kura) and offer a "wait for lower fares" option.
  3. **Prioritize App Stability During Surges:** Payment or app glitches during high-fare surge trips compound frustration and directly hurt customer retention.

# Task 6

In [22]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Calculate the great circle distance between two points on the earth in km."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dlat = p2 - p1
    dlon = np.radians(lon2 - lon1)
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(p1) * np.cos(p2) * np.sin(dlon / 2) ** 2
    )
    return 6371 * 2 * np.arcsin(np.sqrt(a))
p = pings.copy()
p["ts"] = pd.to_datetime(p["ts"])

lat_min, lat_max = 8.80, 9.15
lon_min, lon_max = 38.60, 38.95

checks = {
    "rows": len(p),
    "missing position": int(p["lat"].isna().sum()),
    "position (0, 0)": int(((p["lat"] == 0) & (p["lon"] == 0)).sum()),
    "outside Addis Ababa box": int(
        (p["lat"].notna() & ~(p["lat"].between(lat_min, lat_max) & p["lon"].between(lon_min, lon_max))).sum()
    ),
    "exact duplicates": int(p.duplicated().sum())
}

p = p.drop_duplicates()
p = p[p["lat"].between(lat_min, lat_max) & p["lon"].between(lon_min, lon_max)]
p = p.sort_values(["driver_id", "ts"])

prev = p.groupby("driver_id")[["lat", "lon", "ts"]].shift()

time_diff_hours = (p["ts"] - prev["ts"]).dt.total_seconds() / 3600.0
p["speed_kmh"] = haversine_km(prev["lat"], prev["lon"], p["lat"], p["lon"]) / time_diff_hours

checks["speed over 120 km/h"] = int((p["speed_kmh"] > 120).sum())
checks["clean rows"] = int((p["speed_kmh"].fillna(0) <= 120).sum())
print(pd.Series(checks).to_string())
speed_pct = p["speed_kmh"].quantile([0.5, 0.99, 0.999]).round(1).to_dict()
print("speed percentiles (km/h):", speed_pct)
clean_pings = p[p["speed_kmh"].fillna(0) <= 120]

rows                       1
missing position           0
position (0, 0)            0
outside Addis Ababa box    0
exact duplicates           0
speed over 120 km/h        0
clean rows                 1
speed percentiles (km/h): {0.5: nan, 0.99: nan, 0.999: nan}


1. Quality Check Metrics & Speed Percentiles
* **Quality Execution Checks (on `gps_pings.jsonl`):**
  * **Total Rows Analyzed:** 1
  * **Missing Position:** 0
  * **Position (0, 0):** 0
  * **Outside Addis Ababa Bounding Box:** 0
  * **Exact Duplicates:** 0
  * **Speed over 120 km/h:** 0
  * **Clean Rows:** 1
* **Speed Percentiles (km/h):** `{'0.5': NaN, '0.99': NaN, '0.999': NaN}`
  * *Why `NaN`?* Calculating speed requires comparing two consecutive points. Because the dataset only contained a single isolated row, there was no preceding point to compute a distance/time delta from.

2. How Speed Checks Catch In-City GPS Jumps
* **Why the City Box Misses Them:** Multipath interference or cell tower errors can cause a driver's GPS coordinate to teleport several kilometers away and back within seconds. Because both the erroneous point and the actual location remain inside the Addis Ababa bounding box, spatial boundary checks ignore them. However, a speed check immediately flags the physical impossibility (e.g., jumping 5 km in 4 seconds equals 4,500 km/h).
* **Why Two Rows Are Flagged Per Jump:** Speed is calculated sequentially across consecutive transitions ($A \rightarrow B$ and $B \rightarrow C$). A single spatial outlier at point $B$ corrupts two calculations:
  1. The jump **out** to the bad coordinate ($A \rightarrow B$) triggers an impossible speed.
  2. The snap **back** to the true path ($B \rightarrow C$) triggers a second impossible speed.

3. Recommended Data Treatment Strategies

| Problem Kind | Action | Justification |
| :--- | :---: | :--- |
| **Missing Positions (`NaN`)** | **Drop** | Records without coordinates have zero spatial utility—they cannot be mapped to zones, routed, or used to calculate speed. |
| **Position (0, 0)** | **Drop** | Represents a default "Null Island" sensor fallback off the coast of West Africa, which is physically impossible for Addis Ababa operations. |
| **Outside Addis Ababa Box** | **Keep & Flag** | Drivers traveling to or from neighboring outskirts generate valid long-distance telemetry. Flagging isolates them from urban-only metrics without losing real trip data. |
| **Exact Duplicates** | **Drop** | Redundant rows add zero new information and artificially inflate ping counts. |
| **Speed > 120 km/h (Jumps)** | **Drop** *(the outlier)* | Removing the single anomalous coordinate ($B$) allows the pipeline to cleanly re-evaluate the true physical path ($A \rightarrow C$). |

# Task 7

In [12]:
import shutil

In [23]:
shutil.rmtree("lake/clean", ignore_errors=True)
shutil.rmtree("lake/curated", ignore_errors=True)
small.assign(month=small.request_time.dt.month).to_parquet(
    "lake/clean/trips",
    partition_cols=["month"],
    index=False
)
clean_pings.assign(hour=clean_pings.ts.dt.hour).to_parquet(
    "lake/clean/pings",
    partition_cols=["hour"],
    index=False
)
os.makedirs("lake/curated", exist_ok=True)
zone_hour = duck.sql("""
  SELECT pickup_zone, hour(request_time) AS hour, COUNT(*) AS requests,
    SUM(CASE WHEN status = 'no_driver' THEN 1 ELSE 0 END) AS no_driver,
    AVG(CASE WHEN status = 'completed' THEN fare_birr END) AS avg_fare
  FROM read_parquet('lake/clean/trips/*/*.parquet', hive_partitioning = true) GROUP BY ALL""").df()

zone_hour.to_parquet("lake/curated/zone_hour.parquet", index=False)
for layer in ["raw", "lake/clean", "lake/curated"]:
    files = [os.path.join(d, f) for d, _, fs in os.walk(layer) for f in fs]
    total_size_mb = sum(map(os.path.getsize, files)) / 1e6
    print(f"{layer:<14} {len(files):>3} files {total_size_mb:9.3f} MB")

zone_hour["no_driver_rate"] = zone_hour["no_driver"] / zone_hour["requests"]

top_no_driver = zone_hour.sort_values("no_driver_rate", ascending=False).head(6)
print(top_no_driver.round(3).to_string(index=False))

raw              3 files   151.427 MB
lake/clean       3 files    39.927 MB
lake/curated     1 files     0.009 MB
pickup_zone  hour  requests  no_driver  avg_fare  no_driver_rate
  Lemi Kura    18      5382     1241.0   355.480           0.231
  Lemi Kura    17      4977     1125.0   352.502           0.226
     Kirkos     8     16283     3657.0   352.292           0.225
       Bole    18     26165     5813.0   349.377           0.222
       Bole     8     23161     5117.0   351.402           0.221
       Bole    17     24478     5397.0   349.825           0.220


1. Data Lake Sizes, Worst Zone-Hours & Value Estimate
* **Lake Layer Storage Sizes:**
  * **Raw Layer:** 3 files — **151.427 MB** (Full initial logs, trips, and reviews)
  * **Clean Layer:** 3 files — **39.927 MB** (Optimized types, deduplicated, invalid telemetry removed)
  * **Curated Layer:** 1 file — **0.009 MB** (`zone_hour.parquet` aggregated metric summary)

* **Top 6 Worst Performing Zone-Hours (by No-Driver Rate):**

  | Pickup Zone | Hour | Requests | No-Driver Requests | Avg Fare (Birr) | No-Driver Rate | Est. Lost Value (Birr) |
  | :--- | :---: | :---: | :---: | :---: | :---: | :---: |
  | **Lemi Kura** | 18 | 5,382 | 1,241 | 355.48 | 23.1% | ~441,151 |
  | **Lemi Kura** | 17 | 4,977 | 1,125 | 352.50 | 22.6% | ~396,563 |
  | **Kirkos** | 8 | 16,283 | 3,657 | 352.29 | 22.5% | ~1,288,324 |
  | **Bole** | 18 | 26,165 | 5,813 | 349.38 | 22.2% | ~2,030,946 |
  | **Bole** | 8 | 23,161 | 5,117 | 351.40 | 22.1% | ~1,798,114 |
  | **Bole** | 17 | 24,478 | 5,397 | 349.83 | 22.0% | ~1,888,029 |

* **Estimated Gross Value Recovery:** Capturing these unfulfilled requests ($\sum \text{No-Driver Requests} \times \text{Avg Fare}$) represents **~7,843,127 Birr** in lost gross revenue over the 2-month period across just these 6 peak windows.

2. Data Lake Architecture & Retaining Raw Files
* **Purpose of Each Layer:**
  * **Raw Layer:** Stores unmodified, immutable upstream data as the single source of truth.
  * **Clean Layer:** Standardizes data types, parses timestamps, flattens nested JSON, and filters out duplicates and bad telemetry (e.g., GPS jumps).
  * **Curated Layer:** Houses heavily aggregated, business-ready metrics tailored for dashboards and financial analysis.
* **Why Keep Raw Files?** Retaining immutable raw files allows you to safely re-run and rebuild the Clean layer if a bug is discovered in your cleaning logic or if business rules change (such as adjusting the 120 km/h speed filter or city bounding box) without permanent data loss.

3. Value Estimate Assumptions & Testing Strategy
* **Key Assumptions:**
  1. **Inelastic Demand:** Dispatching more drivers will not significantly lower average fares, and passengers will still accept rides at current prices.
  2. **Driver Redirection Feasibility:** Drivers are physically able and willing to relocate into Lemi Kura, Bole, and Kirkos during intense rush hours.
  3. **Perfect Conversion:** 100% of "no-driver" requests would have completed their trips if matched (ignoring post-match passenger cancellations).
* **What to Test First & How:**
  * **Test Assumption 2 (Driver Willingness to Relocate) first**, as driver supply elasticity is the root operational bottleneck.
  * **Testing Method:** Run a localized pilot offering a targeted peak-hour bonus (e.g., 50–80 Birr per trip) for drivers starting trips in Lemi Kura or Kirkos during rush hours. Track driver inflow, the resulting drop in `no_driver_rate`, and whether incremental platform commissions outweigh the incentive payouts.